# DriverGuard — train mắt CEW V3
Phân loại CLOSED=0, OPEN=1. Chỉ giữ quy trình đã dùng cho CEW V3, không còn model cũ hoặc webcam. Model trạng thái mắt khác với logic đếm thời gian nhắm và cảnh báo.
Chỉ chạy theo thứ tự khi cần train lại; app sử dụng model đã lưu.


## 1. Kết nối Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Đọc ảnh CEW và tọa độ tâm mắt
ClosedFace / OpenFace; hiển thị chấm để kiểm tra tọa độ.

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import random

CEW_ROOT = Path(
    "/content/drive/MyDrive/DriverGuard/CEW/CEW/cropped/dataset_B_FacialImages"
)

def read_eye_coordinates(txt_path):
    coordinates = {}

    with open(txt_path, "r", encoding="utf-8", errors="ignore") as file:
        for line in file:
            parts = line.strip().split()

            if len(parts) >= 5:
                filename = parts[0]
                values = list(map(int, parts[1:5]))
                coordinates[filename] = values

    return coordinates


closed_coordinates = read_eye_coordinates(
    CEW_ROOT / "EyeCoordinatesInfo_ClosedFace.txt"
)

open_coordinates = read_eye_coordinates(
    CEW_ROOT / "EyeCoordinatesInfo_OpenFace.txt"
)

print("Tọa độ mắt ảnh nhắm:", len(closed_coordinates))
print("Tọa độ mắt ảnh mở:", len(open_coordinates))


def show_samples(class_name, coordinates, number_of_images=5):
    image_folder = CEW_ROOT / class_name

    valid_names = [
        name for name in coordinates
        if (image_folder / name).exists()
    ]

    selected_names = random.sample(
        valid_names,
        min(number_of_images, len(valid_names))
    )

    plt.figure(figsize=(15, 4))

    for index, filename in enumerate(selected_names):
        image = Image.open(image_folder / filename).convert("RGB")
        draw = ImageDraw.Draw(image)

        x1, y1, x2, y2 = coordinates[filename]

        # Hai vòng tròn đánh dấu tâm của hai mắt
        radius = 3
        draw.ellipse(
            (x1-radius, y1-radius, x1+radius, y1+radius),
            fill="red"
        )
        draw.ellipse(
            (x2-radius, y2-radius, x2+radius, y2+radius),
            fill="lime"
        )

        plt.subplot(1, number_of_images, index + 1)
        plt.imshow(image)
        plt.title(class_name)
        plt.axis("off")

    plt.tight_layout()
    plt.show()


show_samples("ClosedFace", closed_coordinates)
show_samples("OpenFace", open_coordinates)

## 3. Cắt hai mắt
Giữ nguyên crop lúc train: đệm ngang 0.45, dọc 0.35 lần khoảng cách tâm mắt; RGB 128×48.

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import random
import numpy as np

TARGET_SIZE = (128, 48)


def calculate_eye_box(image_size, eye_coordinates):
    image_width, image_height = image_size
    x1, y1, x2, y2 = eye_coordinates

    eye_distance = max(
        np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2),
        1
    )

    left = int(min(x1, x2) - 0.45 * eye_distance)
    right = int(max(x1, x2) + 0.45 * eye_distance)
    top = int(min(y1, y2) - 0.35 * eye_distance)
    bottom = int(max(y1, y2) + 0.35 * eye_distance)

    # Không cho khung vượt khỏi ảnh
    left = max(0, left)
    top = max(0, top)
    right = min(image_width, right)
    bottom = min(image_height, bottom)

    return left, top, right, bottom


def crop_eye_region(image, eye_coordinates):
    box = calculate_eye_box(image.size, eye_coordinates)
    eye_region = image.crop(box)
    eye_region = eye_region.resize(TARGET_SIZE)

    return eye_region, box


def preview_eye_crops(class_name, coordinates, number_of_images=8):
    folder = CEW_ROOT / class_name

    valid_names = [
        name for name in coordinates
        if (folder / name).exists()
    ]

    selected_names = random.sample(
        valid_names,
        min(number_of_images, len(valid_names))
    )

    plt.figure(figsize=(16, 7))

    for index, filename in enumerate(selected_names):
        face_image = Image.open(folder / filename).convert("RGB")

        eye_image, box = crop_eye_region(
            face_image,
            coordinates[filename]
        )

        face_with_box = face_image.copy()
        draw = ImageDraw.Draw(face_with_box)
        draw.rectangle(box, outline="yellow", width=2)

        # Hàng trên: ảnh mặt và khung cắt
        plt.subplot(2, number_of_images, index + 1)
        plt.imshow(face_with_box)
        plt.title(class_name)
        plt.axis("off")

        # Hàng dưới: kết quả vùng mắt
        plt.subplot(2, number_of_images, number_of_images + index + 1)
        plt.imshow(eye_image)
        plt.title("Eye crop")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


preview_eye_crops("ClosedFace", closed_coordinates)
preview_eye_crops("OpenFace", open_coordinates)

## 4. Đọc ảnh và chia tập
70% train / 15% validation / 15% test; stratify theo nhãn, seed=42. RGB uint8 0–255, chuẩn hóa trong model.

In [ ]:
import json
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

tf.keras.utils.set_random_seed(42)

IMAGE_WIDTH = 128
IMAGE_HEIGHT = 48


# =========================================================
# 1. ĐỌC VÀ CẮT VÙNG HAI MẮT
# =========================================================

def load_cew_class(class_name, coordinates, label):
    folder = CEW_ROOT / class_name

    images = []
    labels = []
    skipped = 0

    for filename, eye_points in coordinates.items():
        image_path = folder / filename

        if not image_path.exists():
            skipped += 1
            continue

        try:
            face_image = Image.open(image_path).convert("RGB")

            eye_image, _ = crop_eye_region(
                face_image,
                eye_points
            )

            eye_array = np.asarray(
                eye_image.resize((IMAGE_WIDTH, IMAGE_HEIGHT)),
                dtype=np.uint8
            )

            images.append(eye_array)
            labels.append(label)

        except Exception as error:
            skipped += 1
            print(f"Bỏ qua {filename}: {error}")

    print(
        f"{class_name}: {len(images)} ảnh hợp lệ, "
        f"{skipped} ảnh bị bỏ qua"
    )

    return images, labels


closed_images, closed_labels = load_cew_class(
    "ClosedFace",
    closed_coordinates,
    label=0
)

open_images, open_labels = load_cew_class(
    "OpenFace",
    open_coordinates,
    label=1
)

X = np.array(closed_images + open_images, dtype=np.uint8)
y = np.array(closed_labels + open_labels, dtype=np.float32)

print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)
print("CLOSED:", int(np.sum(y == 0)))
print("OPEN:", int(np.sum(y == 1)))


# =========================================================
# 2. CHIA TRAIN / VALIDATION / TEST
# 70% train, 15% validation, 15% test
# =========================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nTRAIN:", len(X_train))
print("VALIDATION:", len(X_validation))
print("TEST:", len(X_test))




## 5. CNN V3
Conv 32/64/128, Flatten, Dense 64, Dropout 0.3, sigmoid; không augmentation/BatchNormalization. Adam 0.0003, binary crossentropy.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)


# =========================================================
# MODEL V3: CNN CƠ BẢN, KHÔNG AUGMENTATION, KHÔNG BATCH NORM
# =========================================================

model_v3 = tf.keras.Sequential(
    [
        tf.keras.layers.Input(
            shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3)
        ),

        tf.keras.layers.Rescaling(1.0 / 255),

        tf.keras.layers.Conv2D(
            32, 3,
            padding="same",
            activation="relu"
        ),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(
            64, 3,
            padding="same",
            activation="relu"
        ),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(
            128, 3,
            padding="same",
            activation="relu"
        ),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Flatten(),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dropout(0.30),

        tf.keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    ],
    name="driverguard_eye_model_v3"
)


model_v3.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0003
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


callbacks_v3 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=0.00001
    )
]



model_v3.summary()


## 6. Train
25 epochs tối đa, batch 32. EarlyStopping theo val_loss và khôi phục trọng số tốt nhất.

In [ ]:
history_v3 = model_v3.fit(
    X_train,
    y_train,
    validation_data=(X_validation, y_validation),
    epochs=25,
    batch_size=32,
    callbacks=callbacks_v3,
    shuffle=True,
    verbose=1
)




## 7. Đánh giá và biểu đồ
Ngưỡng sigmoid 0.5. Precision/recall/F1, confusion matrix, accuracy/loss train và validation.
Chia ngẫu nhiên theo ảnh chưa bảo đảm tách danh tính. Tập test từng được xem trong thử nghiệm V1/V2/V3 nên không phải đánh giá cuối trên tập hoàn toàn độc lập.
Output lần train V3 cũ không có trong bản vừa khôi phục; không chèn số liệu giả. Chạy sau train để tạo kết quả mới.

In [ ]:
# =========================================================
# ĐÁNH GIÁ
# =========================================================

test_loss_v3, test_accuracy_v3 = model_v3.evaluate(
    X_test,
    y_test,
    verbose=0
)

scores_v3 = model_v3.predict(
    X_test,
    verbose=0
).reshape(-1)

predictions_v3 = (
    scores_v3 >= 0.5
).astype(int)

print(f"\nTest loss: {test_loss_v3:.4f}")
print(f"Test accuracy: {test_accuracy_v3:.4f}")

print("\nKhoảng điểm dự đoán:")
print("Min :", float(scores_v3.min()))
print("Mean:", float(scores_v3.mean()))
print("Max :", float(scores_v3.max()))

print("\nClassification report:")
print(
    classification_report(
        y_test.astype(int),
        predictions_v3,
        target_names=["CLOSED", "OPEN"],
        digits=4,
        zero_division=0
    )
)


matrix_v3 = confusion_matrix(
    y_test.astype(int),
    predictions_v3
)

ConfusionMatrixDisplay(
    matrix_v3,
    display_labels=["CLOSED", "OPEN"]
).plot(cmap="Blues")

plt.title("CEW Eye Model V3")
plt.show()


# =========================================================
# BIỂU ĐỒ
# =========================================================

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(
    history_v3.history["accuracy"],
    label="Train"
)
plt.plot(
    history_v3.history["val_accuracy"],
    label="Validation"
)
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(
    history_v3.history["loss"],
    label="Train"
)
plt.plot(
    history_v3.history["val_loss"],
    label="Validation"
)
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()




## 8. Lưu model khi cần
Mặc định không lưu đè. Bật SAVE_RETRAINED_MODEL để lưu tên mới.

In [ ]:
# Bật khi cần lưu lần train mới; không ghi đè model đang dùng.
SAVE_RETRAINED_MODEL = False
if SAVE_RETRAINED_MODEL:
    model_folder = Path("/content/drive/MyDrive/DriverGuard/models")
    model_folder.mkdir(parents=True, exist_ok=True)
    model_v3_path = model_folder / "eye_model_cew_v3_retrained.keras"
    model_v3.save(model_v3_path)
    print("Đã lưu:", model_v3_path)
else:
    print("Chưa lưu lại. File eye_model_cew_v3.keras hiện có vẫn được giữ.")
